In [2]:
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv('utube_heat_dataset_ml.csv')

# 2. Подготовка признаков (X) и целевой переменной (y)
# Колонка 'tag' - это уникальный идентификатор (например, P11, P12), 
# для обучения модели он не нужен, поэтому дропаем его вместе с таргетом.
X = df.drop(columns=['tag', 'weight_kg'])
y = df['weight_kg']

# 3. Разделение на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Построение ML пайплайна
# StandardScaler приведет фичи к нулевому среднему и единичной дисперсии. 
# Для деревьев это не строго обязательно, но полезно для стабильности и если потом 
# решишь переключиться на линейные модели или нейросети (MLP).
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# 5. Обучение модели
pipeline.fit(X_train, y_train)

# 6. Валидация и оценка качества
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== Результаты оценки модели на тестовой выборке ===")
print(f"MAE (Средняя абсолютная ошибка): {mae:.2f} кг")
print(f"RMSE (Корень из среднеквадратичной ошибки): {rmse:.2f} кг")
print(f"R2 Score (Коэффициент детерминации): {r2:.4f}")

# 7. Анализ важности признаков (Feature Importance)
# Полезно для понимания физики: какие параметры больше всего влияют на массу
importances = pipeline.named_steps['model'].feature_importances_
feature_names = X.columns

print("\n=== Важность признаков ===")
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(importance_df.to_string(index=False))

filename = 'utube_weight_model.joblib'
joblib.dump(pipeline, filename)

print(f"\n[OK] Модель успешно сохранена в файл: {filename}")

=== Результаты оценки модели на тестовой выборке ===
MAE (Средняя абсолютная ошибка): 62.01 кг
RMSE (Корень из среднеквадратичной ошибки): 118.84 кг
R2 Score (Коэффициент детерминации): 0.9996

=== Важность признаков ===
          Feature  Importance
        heat_area    0.900354
   shell_diameter    0.088354
tube_out_diameter    0.005467
    tube_des_pres    0.003990
         tube_len    0.001835

[OK] Модель успешно сохранена в файл: utube_weight_model.joblib
